# 64. The rest of the frozen-fold library, thirty-two more

**One variable against ledger row 147** (`stack_prune65_library`, CV 0.969737): the member set.
The 32 members of srcL's library not taken in row 147, all covered by the same
transitivity proof: `pub_rmlp` and `pub_tabm` from that library are bit-identical to vectors this
repo verified independently through their authors' printed per-fold AUCs.

These are the weaker two thirds of the library, 0.9585 to 0.9666, including its documented
failures. Row 147 took the top 34 by out-of-fold AUC; this takes what is left.

## A probe that failed, recorded because it decided what is NOT in this run

`writeup/partition_probe.py` tried to detect a foreign fold partition without needing the
author's printed AUCs, by measuring whether a member's per-fold calibration varies the way five
distinct fold-models should. It was calibrated against ground truth in both directions: thirteen
members verified on our partition, and `lookup_srcA` and `spline_srcD`, both proven foreign.

**It does not separate.** Lowest known-good intercept spread 0.00458, highest known-bad 0.02477.
One detail is suggestive, `lookup_srcA`'s slope spread of 0.00061 sits an order of magnitude below
every good member, which is what a mixture should look like, but with two known-bad examples the
test is underpowered and a threshold fitted to two points is not a threshold.

**The consequence is what matters.** `srcJ`'s eleven-model and `srcM`'s seven-model
libraries both document the identical frozen scheme, and srcK's README says outright that it
exists to be stackable with srcL's. Both are excluded from this run, because a README is a claim
and the probe that might have promoted it to a measurement failed. They are downloaded and
available if a verification route appears.

## The case against

**The offset, again.** +0.001266 own models, +0.001259 at five public, +0.001224 at ten. Row 147's
realised offset is not yet known at the time of writing. These 32 are weaker members and several
are srcL's documented failures, so they add little signal and can only dilute.

**Row 142's bound.** `logreg`, `knn` and the `view_*` failures sit far below the field. This repo
measured a member 0.0155 behind contributing exactly zero, and srcL's own manifest records the
eight `view_*` models being worth +0.000007 to his blend between them.

## The prediction

**+0.00005 to +0.00020.** Weaker members, heavily overlapping families, offered to a stack that
already holds ninety-nine vectors. Top 10 percent needs about +0.00007 CV over row 147 at the
current offset, so this should reach it on CV while the leaderboard is the open question.


In [1]:
# 37_stack_views.ipynb
# Membership gate for the five vectors produced by notebooks 35 and 36.
# Runs locally: every member vector already lives in artifacts/oof.
import hashlib
import json
import pathlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(b for b in [Path.cwd(), *Path.cwd().parents]
            if (b / "data" / "raw" / "train.csv").exists())
O = ROOT / "artifacts" / "oof"
S = ROOT / "submissions"

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy(np.int8)

# The fold vector is rebuilt rather than loaded, and then checked. A silently different
# fold vector is the one error here that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
FOLD_SHA = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
assert FOLD_SHA == "ec282b0968059676", FOLD_SHA
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")
print(f"fold sha {FOLD_SHA}  VERIFIED")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]
fold sha ec282b0968059676  VERIFIED


In [2]:
# Row 94's forty-one minus the duplicate, in row 94's order, then the five candidates.
BASE = [
    ("te42", "te_bag42"), ("te2024", "te_seed2024"), ("te7", "te_seed7"),
    ("te2025", "te_seed2025"), ("te13", "te_seed13"),
    ("anchor", "lgbm_default_anchor_seed42"), ("trees300", "lgbm_trees300_seed42"),
    ("trees1000", "lgbm_trees1000_seed42"), ("trees2000", "lgbm_trees2000_seed42"),
    ("lr010", "lgbm_lr01_n1000_seed42"), ("lr005", "lgbm_lr005_n2000_seed42"),
    ("lr003", "lgbm_lr003_n3333_seed42"),
    ("bag42", "lgbm_bag08_lr005_n2000_seed42"),
    ("bag2024", "lgbm_bag08_lr005_n2000_seed2024"),
    ("bag7", "lgbm_bag08_lr005_n2000_seed7"),
    ("bag2025", "lgbm_bag08_lr005_n2000_seed2025"),
    ("bag13", "lgbm_bag08_lr005_n2000_seed13"),
    ("neural", "neural"), ("cat42", "catboost_te"), ("cat2024", "catboost_te_seed2024"),
    ("cat7", "catboost_te_seed7"), ("cat2025", "catboost_te_seed2025"),
    ("cat13", "catboost_te_seed13"), ("neural_te", "neural_te"),
    ("xgb_te", "xgb_te"), ("xgb2024", "xgb_te_seed2024"), ("xgb7", "xgb_te_seed7"),
    ("xgb2025", "xgb_te_seed2025"), ("xgb13", "xgb_te_seed13"),
    ("pair_top9", "xgb_pair_top9"),
]
BASE += [("cat_nat_c1", "cat_native_c1"), ("cat_nat_c2", "cat_native_c2"),
         ("xgb_raw", "xgb_raw"), ("cat_raw", "cat_raw"),
         ("xgb_te_fe", "xgb_te_fe"), ("cat_te_n4000", "cat_te_n4000"),
         ("xgb_raw_fe", "xgb_raw_fe"), ("cat_raw_n10k", "cat_raw_n10000"),
         ("lgb_raw_fe", "lgb_raw_fe"), ("cat_raw_fe", "cat_raw_fe")]
# lgb_raw is deliberately absent: it is bag42 re-run in another kernel, Pearson 0.999970 on
# logits. See the header and the 2026-08-22 entry. Dropping it costs -0.000001.
DROPPED = [("lgb_raw", "duplicate of bag42, Pearson 0.999970")]

BASE += [("cat_te_fe", "cat_te_fe"), ("hgb_te_fe", "hgb_te_fe"),
         ("lgb_te_fe", "lgb_te_fe"), ("rf_te_fe", "rf_te_fe"),
         ("neural_lookup", "neural_lookup"), ("et_te_fe", "et_te_fe"),
         ("logit_te_fe", "logit_te_fe")]
BASE += [("xgb_tuned", "xgb_tuned")]
BASE += [("neural_fe", "neural_fe"), ("neural_wide", "neural_wide"),
         ("neural_res", "neural_res")]
BASE += [("realmlp", "realmlp")]
BASE += [("realmlp10", "realmlp10")]
BASE += [("realmlp_raw_fe", "realmlp_raw_fe")]
BASE += [("tabm", "tabm")]
# THE ONE VARIABLE. Row 144 held these fifty-five, all built by this repo. The five
# candidates below were not. Each is admitted only by writeup/verify_public_oof.py,
# which proves the fold partition matches ours by reproducing the author's own printed
# per-fold AUCs from our fold vector. See the header.
# Row 145's five, now part of the base.
PRIOR_PUBLIC = ["cb_srcE", "lgb_srcE", "xgb_srcA", "realmlp_srcA", "tabm_srcA",
                "lgb_srcB", "realmlp_srcI", "hgb_srcH", "xgb_srcC", "resnet_kava"]
PUBLIC = PRIOR_PUBLIC
# THE ONE VARIABLE. Thirty-four members from the frozen-fold library, loaded as raw
# .npy rather than through the kernel gate, because their provenance is established by
# transitivity against pub_rmlp and pub_tabm being bit-identical. See the header.
PRIOR_SZY = sorted((ROOT / "artifacts" / "wide_library" / "picked.txt").read_text().split())
_szy_raw = sorted((ROOT / "artifacts" / "wide_library" / "picked2.txt").read_text().split())
# This batch contains `realmlp` and `xgb_tuned`, which collide with OUR member aliases.
# Namespaced so the loader cannot silently overwrite one of ours.
SZY = [f"szy_{n}" for n in _szy_raw]
_SZY_FILE = {f"szy_{n}": n for n in _szy_raw}
CAND = [(n, n) for n in SZY]


def load(stem, kind):
    # OOF/test vector. Two naming conventions exist in artifacts/oof. The bare
    # "{stem}.npy" form is the OOF side only: the early LightGBM members never had a
    # test .npy written and their test side lives in submissions/. Falling back to the
    # bare name for kind="test" silently returns the OOF vector, which is caught by the
    # length assert below only because train and test differ in length.
    cands = [O / f"{stem}_{kind}.npy"]
    if kind == "oof":
        cands.append(O / f"{stem}.npy")
    for c in cands:
        if c.exists():
            return np.load(c)
    if kind == "test" and (S / f"{stem}.csv").exists():
        df = pd.read_csv(S / f"{stem}.csv")
        # A csv written in a different row order blends perfectly cleanly and is
        # undetectable in the score. Checked rather than assumed.
        assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {stem}"
        return df["addicted_label"].to_numpy()
    raise FileNotFoundError(f"{stem} {kind}")


# Row 145 already holds PRIOR_PUBLIC, so they belong in the index alongside our own.
# Row 147 holds BASE + PRIOR_PUBLIC + PRIOR_SZY, so all three belong in the index.
MEM = BASE + [(n, n) for n in PRIOR_PUBLIC] + [(n, n) for n in PRIOR_SZY] + CAND
names = [n for n, _ in MEM]
Poof = {n: load(s, "oof") for n, s in BASE}
Ptest = {n: load(s, "test") for n, s in BASE}

# The public five, read from artifacts/public_oof/ with their id order asserted rather
# than assumed. A csv in a different row order blends perfectly cleanly and is invisible
# in the score, which is the failure row 59's loader already guards against.
PUB = ROOT / "artifacts" / "public_oof"
VER = {r["name"]: r for r in json.loads((PUB / "verification.json").read_text(encoding="utf-8"))}


def read_vec(path, n_expected, order_ref):
    """Mirror of writeup/verify_public_oof.load_vector, so a member is loaded here
    exactly as it was loaded when it was verified. Two shapes exist in the wild: a
    csv with or without an id column, and a bare .npy."""
    path = pathlib.Path(path)
    if path.suffix == ".npy":
        v = np.load(path)
        assert len(v) == n_expected, f"{path.name} has {len(v)} rows"
        return v.astype(float)
    df = pd.read_csv(path)
    assert len(df) == n_expected, f"{path.name} has {len(df)} rows"
    idc = [c for c in df.columns if c.lower() == "id"]
    if idc:
        # A csv in a different row order blends perfectly cleanly and is invisible in
        # the score, so this is asserted rather than hoped for.
        assert (df[idc[0]].to_numpy() == order_ref).all(), f"id order {path.name}"
    pref = [c for c in df.columns if any(k in c.lower() for k in ("oof", "pred", "prob"))]
    col = pref or [c for c in df.columns
                   if c.lower() not in ("id", "addicted_label", "target", "fold")]
    col = col or [c for c in df.columns if c.lower() != "id"]
    return df[col[0]].to_numpy(float)


for n in PUBLIC:
    r = VER.get(n)
    assert r and r["verdict"].startswith("ADMISSIBLE"), \
        f"{n} is not admissible: {r['verdict'] if r else 'absent'}. Re-run the gate."
    d = PUB / r["folder"]
    Poof[n] = read_vec(d / r["oof_file"], len(train), train["id"].to_numpy())
    Ptest[n] = read_vec(d / r["test_file"], len(test), test["id"].to_numpy())
# The library members. The manifest AUC is asserted, so a truncated or wrong file
# cannot enter quietly.
import csv as _csv
SZD = ROOT / "artifacts" / "wide_library"
_man = {r["model"]: float(r["oof_auc"])
        for r in _csv.DictReader((SZD / "manifest.csv").open(encoding="utf-8"))}
for n in PRIOR_SZY + SZY:
    stem = _SZY_FILE.get(n, n)
    o = np.load(SZD / f"oof_{stem}.npy")
    t = np.load(SZD / f"test_{stem}.npy")
    assert o.shape == (len(train),) and t.shape == (len(test),), n
    _a = roc_auc_score(y, o)
    assert abs(_a - _man[stem]) < 5e-5, f"{n}: AUC {_a:.6f} vs manifest {_man[stem]}"
    Poof[n], Ptest[n] = o.astype(float), t.astype(float)
print(f"{len(PRIOR_SZY)} carried + {len(SZY)} new library members, all matching manifest AUC")
print(f"{len(PUBLIC)} kernel-verified public members loaded")
print(f"  kernel-verified     : {len(PRIOR_PUBLIC)}")
print(f"  library candidates  : {len(SZY)}")
_rej = [k for k, v in VER.items() if not v["verdict"].startswith("ADMISSIBLE")]
print(f"  rejected by the gate: {_rej}")

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    assert np.isfinite(Poof[n]).all() and np.isfinite(Ptest[n]).all(), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

# Exact-duplicate quarantine. A duplicated array silently DOUBLES that model's weight.
# Row 59 found xgb_pair_base bit-identical to xgb_te this way and excluded it.
seen = {}
for n in names:
    h = hashlib.md5(np.ascontiguousarray(Poof[n]).tobytes()).hexdigest()
    assert h not in seen, f"{n} is bit-identical to {seen[h]}"
    seen[h] = n
print(f"{len(names)} vectors loaded, no exact duplicates")

# THE CORRELATION SCREEN. Hash equality cannot see one configuration run in two kernels:
# the arrays differ by thread-level numerical noise. bag42 and lgb_raw sat in row 94 at
# Pearson 0.999970 and no check fired. Three lines, and it would have caught them.
Lz = np.column_stack([np.clip(np.log(np.clip(Poof[n], 1e-9, 1 - 1e-9)
                                     / (1 - np.clip(Poof[n], 1e-9, 1 - 1e-9))), -30, 30)
                      for n in names])
Cm = np.corrcoef(((Lz - Lz.mean(0)) / Lz.std(0)).T)
np.fill_diagonal(Cm, 0.0)
near = [(names[i], names[j], Cm[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
        if abs(Cm[i, j]) > 0.9999]
print(f"pairs above 0.9999: {len(near)}")
for a, b, r in near:
    print(f"  NEAR-DUPLICATE {a} and {b} at {r:+.6f}")
assert not near, "a near-duplicate pair is present, justify it or drop one"
print(f"removed from row 94's set: {[d[0] for d in DROPPED]}")
hi = sorted(((abs(Cm[i, j]), names[i], names[j])
             for i in range(len(names)) for j in range(i + 1, len(names))),
            reverse=True)[:3]
print("most collinear surviving pairs: "
      + ", ".join(f"{a}/{b} {r:.5f}" for r, a, b in hi))

34 carried + 33 new library members, all matching manifest AUC
10 kernel-verified public members loaded
  kernel-verified     : 10
  library candidates  : 33
  rejected by the gate: ['lookup_srcA', 'spline_srcD']


132 vectors loaded, no exact duplicates


pairs above 0.9999: 0
removed from row 94's set: ['lgb_raw']
most collinear surviving pairs: realmlp/realmlp10 0.99948, rmlp_lat/rmlp_lat3 0.99942, cat42/cat7 0.99907


In [3]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
IDX = {n: i for i, n in enumerate(names)}
BASE99 = ([IDX[n] for n, _ in BASE] + [IDX[n] for n in PRIOR_PUBLIC]
          + [IDX[n] for n in PRIOR_SZY])

print("candidate solo CV, and disagreement with the members it most resembles:")
print(f"  {'candidate':12} {'solo CV':>10} {'rho vs xgb_te':>15} {'rho vs cat42':>14}")
for n, _ in CAND:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    r1 = pd.Series(Poof[n]).corr(pd.Series(Poof["xgb_te"]), method="spearman")
    r2 = pd.Series(Poof[n]).corr(pd.Series(Poof["cat42"]), method="spearman")
    print(f"  {n:12} {cv:10.6f} {r1:15.6f} {r2:14.6f}")

# For scale: how decorrelated are two members that everyone agrees are near-copies?
r_seed = pd.Series(Poof["xgb_te"]).corr(pd.Series(Poof["xgb2024"]), method="spearman")
print(f"\n  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): {r_seed:.6f}")
print("  this repo refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.")
print("  This table is context, not a prediction.")

candidate solo CV, and disagreement with the members it most resembles:
  candidate       solo CV   rho vs xgb_te   rho vs cat42


  szy_altview    0.964740        0.982325       0.975179


  szy_cat        0.962281        0.976675       0.979325


  szy_cat_tuned   0.961634        0.976403       0.976051


  szy_digit_lgbm   0.965913        0.984732       0.978092


...

  szy_view_resid_lgbm   0.955461        0.964096       0.955168


  szy_view_resid_xgb   0.955198        0.963231       0.954830


  szy_xgb        0.964335        0.982707       0.976772


  szy_xgb_tuned   0.965585        0.984597       0.979198



  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): 0.997344
  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.
  This table is context, not a prediction.


In [4]:
def run(cols):
    # Fold-wise logistic combiner. No weight is ever fitted on a row it is scored on.
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    nit = []
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)], y[tr])
        # A non-converged lbfgs fit reads HIGHER than the truth, so convergence is
        # asserted rather than hoped for. Added 2026-08-21 after the public review
        # flagged it; measured at 38 to 40 iterations, so it has never been close.
        nit.append(int(np.max(clf.n_iter_)))
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    assert max(nit) < 2000, f"combiner did not converge, {nit}"
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf, max(nit)


# Thirty-four single-candidate arms would be 34 extra fits for a table nobody acts on.
# The decision here is about the SET, so the set is the arm, plus the one member
# srcL's own manifest singles out as the most decorrelated thing he found.
ARMS = {"99_row147": BASE99}
ARMS["100_all32"] = BASE99 + [IDX[n] for n, _ in CAND]

res = {a: run(cols) for a, cols in ARMS.items()}

# The prune, recomputed inside a committed notebook rather than trusted from the audit's
# scratch script. Keeps the top k members of the WINNING arm by absolute coefficient.
_best_cols = ARMS["100_all32"]
_coef = res["100_all32"][2].mean(axis=0)
_order = np.argsort(-np.abs(_coef))
for _k in (35, 50, 65, 85):
    if _k < len(_best_cols):
        ARMS[f"prune{_k}"] = sorted(_best_cols[i] for i in _order[:_k])
        res[f"prune{_k}"] = run(ARMS[f"prune{_k}"])
per = {a: r[0] for a, r in res.items()}

# Row 54 compared its base arm against 0.968932, which is row 137's PRUNED value rather
# than the 53-member figure. That labelling slip is recorded in row 139's notes and is
# corrected here: the base arm below IS row 139's fifty-three, and 0.968932 is its CV.
ROW147_CV, ROW140_CV = 0.969737, 0.969737
repro = per["99_row147"].mean() - ROW147_CV
print(f"reproduction of row 147: {per['99_row147'].mean():.6f} vs {ROW147_CV:.6f}"
      f"  delta {repro:+.2e}   {'REPRODUCED' if abs(repro) < 1e-4 else 'FAILED'}")
assert abs(repro) < 1e-4, "base arm does not reproduce row 147, do not log this run"
print(f"combiner max n_iter across all arms: {max(r[3] for r in res.values())} of 2000\n")

hdr = " ".join(f"{'fold ' + str(i):>9}" for i in range(5))
print(f"{'arm':14} {hdr} {'mean':>10} {'sd':>9}")
for a in ARMS:
    print(f"{a:14} " + " ".join(f"{v:9.6f}" for v in per[a])
          + f" {per[a].mean():10.6f} {per[a].std():9.6f}")

reproduction of row 147: 0.969726 vs 0.969737  delta -1.13e-05   REPRODUCED
combiner max n_iter across all arms: 95 of 2000

arm               fold 0    fold 1    fold 2    fold 3    fold 4       mean        sd
99_row147       0.969116  0.969817  0.969830  0.970319  0.969547   0.969726  0.000393
100_all32       0.969153  0.969827  0.969843  0.970364  0.969581   0.969754  0.000394
prune35         0.969141  0.969827  0.969865  0.970345  0.969560   0.969748  0.000395
prune50         0.969165  0.969837  0.969892  0.970373  0.969596   0.969772  0.000395
prune65         0.969180  0.969850  0.969880  0.970374  0.969597   0.969776  0.000390
prune85         0.969181  0.969850  0.969861  0.970373  0.969602   0.969773  0.000388


In [5]:
FLOOR_MEAN, FLOOR_FOLDS = 0.00005, 4
# THE LEAK TRIPWIRE. a feature that jumps CV by an implausible amount is a
# leak until proven otherwise. A verified partition should behave like any other member
# set; anything above +0.002 here means the verification missed something.
LEAK_ALARM = 0.002
base_per = per["99_row147"]

print("Paired against row 147's ninety-nine. The gate is >= +0.00005 mean AND >= 4/5 folds.\n")
print(f"{'arm':14} {'paired mean':>13} {'paired sd':>11} {'folds':>7} {'t(4)':>8}  gate")
gate = {}
for a in ARMS:
    if a == "99_row147":
        continue
    d = per[a] - base_per
    wins = int((d > 0).sum())
    sd = d.std(ddof=1)
    t = d.mean() / (sd / np.sqrt(5)) if sd > 0 else float("inf")
    fired = bool(d.mean() >= FLOOR_MEAN and wins >= FLOOR_FOLDS)
    gate[a] = fired
    print(f"{a:14} {d.mean():+13.6f} {sd:11.6f} {wins:5d}/5 {t:8.2f}"
          f"  {'FIRES' if fired else 'under floor'}")
    assert d.mean() < LEAK_ALARM, (
        f"{a} gains {d.mean():+.6f}, above the {LEAK_ALARM} alarm. Treat as a leak and "
        f"re-run writeup/verify_public_oof.py before believing this.")

Paired against row 147's ninety-nine. The gate is >= +0.00005 mean AND >= 4/5 folds.

arm              paired mean   paired sd   folds     t(4)  gate
100_all32          +0.000028    0.000015     5/5     4.07  under floor
prune35            +0.000022    0.000011     5/5     4.61  under floor
prune50            +0.000047    0.000016     5/5     6.53  under floor
prune65            +0.000050    0.000011     5/5     9.96  FIRES
prune85            +0.000048    0.000015     5/5     7.28  under floor


In [6]:
# Coefficients of the best arm, which is where the result actually lives. Row 59's
# lesson: a model that is null on its own can still take a large weight, and where that
# weight comes FROM is the thing worth reading.
best = max((a for a in ARMS if a != "99_row147"), key=lambda a: per[a].mean())
print(f"best arm by CV: {best}   {per[best].mean():.6f}\n")

cols_b, cols_0 = ARMS[best], ARMS["99_row147"]
cb = res[best][2].mean(axis=0)
c0 = res["99_row147"][2].mean(axis=0)
base_map = {names[c]: c0[i] for i, c in enumerate(cols_0)}

rows = []
for i, c in enumerate(cols_b):
    n = names[c]
    was = base_map.get(n, float("nan"))
    rows.append({"member": n, "coef": cb[i], "was": was, "shift": cb[i] - was})
tab = pd.DataFrame(rows).sort_values("coef", ascending=False)
print(tab.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

new = set(n for n, _ in CAND) & set(tab.member)
gained = tab[tab.member.isin(new)]["coef"].sum()
lost = -tab[~tab.member.isin(new)]["shift"].sum()
print(f"\nnew members carry {gained:+.4f} in total")
print(f"the existing fifty-three give up {lost:+.4f} of weight between them")
if abs(gained) > 1e-12:
    print(f"substitution covers {100 * lost / gained:.0f} percent of the new weight")

best arm by CV: prune65   0.969776

                 member    coef     was   shift
             cat_nat_c2 +0.2354 +0.2443 -0.0089
                 lookup +0.2170 +0.2255 -0.0085
           realmlp_srcI +0.1758 +0.1726 +0.0031
           realmlp_srcA +0.1399 +0.1364 +0.0035
              latr1_xgb +0.1385 +0.1212 +0.0173
            tabm_deeper +0.1089 +0.1208 -0.0119
           szy_pubmk_nn +0.1013     NaN     NaN
             pub_tabnet +0.0875 +0.0846 +0.0030
               lgb_srcB +0.0860 +0.0812 +0.0049
           cat_te_n4000 +0.0841 +0.0751 +0.0089
                szy_xgb +0.0809     NaN     NaN
    szy_view_bounds_cat +0.0756     NaN     NaN
            szy_pub_evg +0.0728     NaN     NaN
            latwide_cat +0.0689 +0.0622 +0.0067
          szy_pubfe_xgb +0.0650     NaN     NaN
              tabm_wide +0.0612 +0.0616 -0.0003
               tabm_imp +0.0588 +0.0674 -0.0085
             lattri_xgb +0.0572 +0.0428 +0.0145
         imp_lgbm_tuned +0.0558 +0.0584 -0.0026
    

In [7]:
# Three decisions, kept separate. Bundling them was the error corrected in row 59.
print("1. GATE")
for a, f in gate.items():
    print(f"     {a:14} {'FIRES' if f else 'under floor'}")

print("\n2. MEMBERSHIP")
print("   A sub-floor addition is still kept and logged as negligible: row 32 kept four")
print("   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership")
print("   follows the sign and the fold count, not the floor.")
keep = [a for a in ARMS if a != "99_row147"
        and (per[a] - base_per).mean() > 0
        and int(((per[a] - base_per) > 0).sum()) >= 4]
print(f"   arms positive and >= 4/5 folds: {keep if keep else 'none'}")
print(f"   carried forward: {best} at {per[best].mean():.6f}")

print("\n3. SUBMISSION")
SUB = S / "stack_library2.csv"
# The floor the gate uses, applied to the submission decision too. Row 142 recorded
# that these had different thresholds and that a two-millionth difference wrote a csv.
if per[best].mean() > ROW140_CV + FLOOR_MEAN:
    p = res[best][1].mean(axis=0)
    sub = pd.DataFrame({"id": test["id"].to_numpy(),
                        "addicted_label": (np.argsort(np.argsort(p)) + 0.5) / len(p)})
    assert len(sub) == len(test) and np.isfinite(sub["addicted_label"]).all()
    sub.to_csv(SUB, index=False)
    print(f"   wrote {SUB.name}, {len(sub):,} rows,"
          f" {sub['addicted_label'].nunique():,} distinct")
    print("   AUC reads order only, so the rank transform changes nothing and keeps the")
    print("   file comparable with the earlier stack submissions.")
else:
    print(f"   no submission: best arm {per[best].mean():.6f} does not beat row 140's "
          f"{ROW140_CV:.6f}")

print(f"\nledger lines:\n  name    stack_{best}\n  cv_mean {per[best].mean():.6f}"
      f"\n  cv_std  {per[best].std():.6f}")

1. GATE
     100_all32      under floor
     prune35        under floor
     prune50        under floor
     prune65        FIRES
     prune85        under floor

2. MEMBERSHIP
   A sub-floor addition is still kept and logged as negligible: row 32 kept four
   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership
   follows the sign and the fold count, not the floor.
   arms positive and >= 4/5 folds: ['100_all32', 'prune35', 'prune50', 'prune65', 'prune85']
   carried forward: prune65 at 0.969776

3. SUBMISSION
   no submission: best arm 0.969776 does not beat row 140's 0.969737

ledger lines:
  name    stack_prune65
  cv_mean 0.969776
  cv_std  0.000390
